---
# <div style="text-align: center"> Introduction </div>
---

Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System** class and its sources: the **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) Running <span style="color:blue">**SCOPE**</span> - Part 1: File Structure
7) Running <span style="color:blue">**SCOPE**</span> - Part 2: Execution 
8) Running <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions

---
# <div style="text-align: center"> Tutorial 2: The Computational Workflow</div>
---

<span style="color:blue">**SCOPE**</span> aims at automatizing computational workflows, or tasks. To encode the potential complexity of such workflows, it needs a flexible hierarchy of classes, whose purpose might not be evident to a new user.  

As currently implemented, the hierarchy of classes associated with a computational workflow goes like this:  **Branch** -> **Workflow** -> **Job** -> **Computation**.  

From Bottom to Top:
- <span style="color:green">**COMPUTATION**</span>: Starting from the bottom, a **Computation** is associated with an input, an output and a submission file, and contains the information of this single **Computation** that is **submitted** to a computational cluster. Whether it worked or not, SCOPE takes the results into account to decide the course of action.
- <span style="color:green">**JOB**</span>: A **Job** is a single, or a collection of **Computations** with a specific goal. For instance, a **Job** could be the Single Point Energy evaluation of a given structure. It might work with only one computation, or it might need more than one, for instance if convergence issues arise.  
- <span style="color:green">**WORKFLOW**</span>: A collection of **Jobs** for a given molecule or unit cell in a system (i.e a source) form a **Workflow**. For instance, a **Workflow** could be: (1) Optimization with Cheap DFT Functional, (2) Optimization with Expensive DFT Functional, (3) Hessian evaluation (frequencies). Workflows are connected to the **Source** of a **System** 
- <span style="color:green">**BRANCH**</span>: A **Branch** is a collection of related **Workflows**. For instance, the Workflows for the high-spin and low-spin versions of the same molecule can be put together under the same **Branch**.

In this tutorial, we will open an existing **SYSTEM**, stored in a binary file, and navigate the computational workflow inside.

In [ ]:
import os
from scope.read_write import load_binary

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data"
data_folder = os.path.abspath('.')+'/Data/1-Tutorial_1/'
## Loads the System object from a binary file, provided in the tutorial folder
sys = load_binary(f"{data_folder}ABITEM.npy")

In [ ]:
## All objects in SCOPE have a __repr__ method, so printing shows a summary of the object
print(sys)

In [ ]:
## Shows the branches of a system
print(sys.branches)

## Branch


In [ ]:
## Select a given branch either by selecting it from the list, or searching by name:
found, this_branch = sys.find_branch("Isolated")
print(found)
print(this_branch)

## You'll likely see a WARNING. This is because the branch was created in my computer, and the message is warning that the folders associated with the computations of this branch does not exist in your computer. 
## You can safely ignore this WARNING for the moment. 

In [ ]:
## Workflows are easily accessible. Here there are two workflows, one for the reference high-spin (HS) molecule (ref_hs_mol) and the other for the low-spin (LS) one (ref_ls_mol)
this_branch.workflows

## Workflow 

In [ ]:
## Again, you can select a given workflow either by selecting it from the list, or searching by name of the source:
found, this_workflow = this_branch.find_workflow("ref_hs_mol")
print(found)
print(this_workflow)

In [ ]:
## This workflow has 3 Jobs, the latest being a b3lyp frequency computation
this_workflow.jobs

## Job

In [ ]:
## Again, you can select a given JOB either by selecting it from the list, or searching by name of the keyword:
found, this_job = this_workflow.find_job('b3lyp_opt')
print(found)
print(this_job)

## Alternatively, you can also search by hierarchy number:
found, that_job = this_workflow.find_job(hierarchy=2) ## hierarchy goes from 1 onwards.
print(found)
print(that_job)


In [ ]:
## Both ways you retrieve the same job.
this_job == that_job

In [ ]:
## Notice that jobs have requisites and constrains. This means that, when submitting this job, it won't run unless those in requisites have finished and those in constrains have not finished.
print(this_job.requisites)
print(this_job.constrains)

In [ ]:
## Jobs store the information provided by the user here:
print(this_job.job_data)
## Some entries inside job_data create an attribute in the object, so you can access those data as either:
print(this_job.istate, this_job.job_data.istate) 

In [ ]:
## Some of this information will be discussed below when talking about the computations. An important bit are .istate and .fstate
print(this_job.istate)
print(this_job.fstate)
## States are a way to manage the results of computations. They are discussed in Tutorial 2
## For the moment, suffices to say that computations take the information from an initial state, and output the information to a final state, that can be the same.

## In this case, the job reads the data from the PBE_OPT state (e.g. initial geometry) and the result from a B3LYP optimization is stored in a final state called B3LYP_OPT.
## Because B3LYP_OPT state does not exist when submitting the computation, the new state will be automatically created

## Computation

In [ ]:
## You can access the computations by selecting it from the list, or searching by step and run_number
found, this_comp = this_job.find_computation(step=1, run_number=1) ## both step and run_number go from 1 onwards.
print(found)
print(this_comp)

In [ ]:
## If everything goes well, most jobs will be completed after a single computation, with step=1 and run_number=1. 
print(this_comp.step, this_comp.run_number, this_comp.isgood)
## Larger values for "run_number" are typical if a computation fails to complete the task.
## In such cases multiple computations with increasing run_number values will be created and run.


In [ ]:
## There is only one type of job setup for which "step" can be different than 1. Currently, the setup type of this job is:
print(this_job.job_setup)

## When .setup is "rep_opt", it will run several 'external' steps of geometry optimization until the energy converges. This is relevant in variable-cell geometry optimizations of unit-cells.
## With Quantum Espresso, variable-cell geo-opts computations can finish successfuly, supposedly giving you the minimum energy structure. 
## However, if you re-submit a new computation to continue the previous one, the energy might still go down significanlty. This is related to how Plane-waves react to the change of cell size.


In [ ]:
## The paths of the input and output files are stored. The name of these files (not the extension) is decided here
this_comp.filename

In [ ]:
## The Quantum Chemistry parameters of this computation are stored in .qc_data: 
this_comp.qc_data

In [ ]:
## Some of this parameters are defined by the used in the job_file that specifies which computation must be done, and some other information is completed by the code.
## Basically, it adds some defaults.

In [ ]:
## Once finished, the computation is registered. This means that the output is read, and the relevant information parsed.
## This is done through the "OUTPUT" class. There is one output class for each of the implemented softwares (for the moment, Gaussian and Quantum Espresso)

## To access the output, we must modify the path:
this_comp.out_path = f"{data_folder}ABITEM_ref_hs_mol_opt2_r1.log"

## Now we can read the output
output = this_comp.create_output()
print(output)

In [ ]:
import matplotlib.pyplot as plt

## And now we can parse some relevant data:
energies = output.get_all_energies()
plt.plot(range(len(energies)), energies)
plt.scatter(range(len(energies)), energies)

In [ ]:
from scope.read_write import print_xyz
labels, coordinates = output.get_geometry_last_complete_block()
print_xyz(labels, coordinates)